In [ ]:
%pip install qiskit qiskit-aer

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

# 1. Import the oracle from oracles.py
from oracles import simon_linear_oracle

# 2. Simon's algorithm sampling function
def run_simon(A, shots=40):
    n = A.shape[0]
    backend = Aer.get_backend("aer_simulator")

    Uf = simon_linear_oracle(A).to_gate(label="Uf")

    Y = []

    for _ in range(shots):
        qc = QuantumCircuit(2*n, n)

        # Superposition on input register
        qc.h(range(n))

        # Apply oracle
        qc.append(Uf, range(2*n))

        # Hadamard again on input register
        qc.h(range(n))

        # Measure input register → yields y
        qc.measure(range(n), range(n))

        compiled = transpile(qc, backend)
        result = backend.run(compiled, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]

        # Reverse for correct qubit order
        y = np.array(list(bitstring[::-1]), dtype=int)
        Y.append(y)

    return np.array(Y)


# 3. Compute kernel A
def find_secret_string(A):
    
    # Find s that A s = 0 (mod 2). Brute force for small n.
    n = A.shape[0]
    for guess in range(1, 2**n):
        s = np.array(list(np.binary_repr(guess, width=n))).astype(int)
        if np.all((A @ s) % 2 == 0):
            return s
    return None


# 4. TEST CASE
A = np.array([
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0],  
    [1, 0, 1, 1]
], dtype=int)

print("Matrix A:")
print(A)

# Run Simon and collect Y vectors
Y = run_simon(A, shots=40)
print("\nCollected measurement vectors (Y):")
print(Y)

# Compute the hidden string s from the kernel of A
s = find_secret_string(A)
print("\nRecovered hidden string s:")
print(s)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Matrix A:
[[1 1 0 0]
 [0 1 1 0]
 [0 0 0 0]
 [1 0 1 1]]

Collected measurement vectors (Y):
[[1 1 0 0]
 [0 1 1 1]
 [1 0 1 0]
 [1 1 0 0]
 [1 0 1 0]
 [1 1 0 1]
 [0 0 0 0]
 [1 0 1 0]
 [1 0 1 0]
 [1 1 0 1]
 [1 1 0 0]
 [1 1 0 0]
 [1 0 1 0]
 [0 0 0 1]
 [0 0 0 1]
 [0 1 1 0]
 [0 0 0 0]
 [1 1 0 1]
 [1 1 0 1]
 [1 1 0 0]
 [1 1 0 0]
 [0 1 1 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 1]
 [1 0 1 0]
 [1 1 0 1]
 [0 0 0 0]
 [0 1 1 0]
 [0 1 1 0]
 [1 1 0 0]
 [1 0 1 0]
 [0 1 1 0]
 [1 0 1 1]
 [1 1 0 0]
 [0 0 0 0]
 [0 0 0 1]
 [1 0 1 1]
 [1 1 0 1]
 [1 1 0 0]]

Recovered hidden string s:
[1 1 1 0]


Here, the oracle works by turning the binary matrix-vector multiplication f(x)=Ax mod2 into XOR operations. In GF(2), each output bit y_i is just the XOR of the input bits x_j wherever the matrix entry A_i,j = 1. A CNOT gate performs this operation where its target qubit flips only when the control qubit is 1, so every “1” in the matrix corresponds to adding a CNOT from input qubit j to output qubit n+i. So, when the circuit runs, it leaves the input register untouched but updates the output to y(+)Ax, which is the reversible version of computing Ax. Therefore, by chaining together these CNOTs within the structure of the matrix, the circuit can now implement a full linear transformations using only quantum XOR gates.

Here, each run of the quantum circuit yields a bitstring y such that y.s=0 (mod2).These equations come from the interference pattern created when the input register is put into superposition and when the oracle maps both x and x(+)s to the same output, forcing the final state (after Hadamards) to contain only vectors that are orthogonal to the secret string. Each measurement therefore gives you one linear equation in the fomr of y_1.s_1 (+) y_2.s_2 ... (+) y_b.s_n =0. For example, if the circuit outputs y=[0,1,1,0], this corresponds to s_2 (+) s_3 = 0. Computing these outputs gives us a simple system of equations over GF(2), and solving this system gives you the unique non-zero vector s that is consistent with all constraints. Essentially, you need about n−1 linearly independent equations to identify s (see below). Therefore, the number of measurements you needed was precisely the number required to span an (n−1)-dimensional subspace orthogonal to s.

Result analysis:

Distinct, non-zero vectors:
[1,1,0,0]
[0,1,1,1]
[1,0,1,0]
[1,1,0,1]
[0,0,0,1]
[0,1,1,0]
[1,0,1,1]
[0,0,0,0] (essentially useless)

(I did this process by hand, but basically...)
equations:
1. s_1 (+) s_2 = 0 
2. s_2 (+) s_3 (+) s_4 = 0 
3. s_1 (+) s_3 = 0
4. s_1 (+) s_2 (+) s_4 = 0
5. s_4 = 0
6. s_2 (+) s_3 = 0
7. s_1 (+) s_3 (+) s_4 = 0
8. 0 = 0

so,
s_4 = 0 (fifth equation)
s_1 = s_2 = s_3 (first, third, sixth equation)

2 solutions:
i) s_1 = s_2 = s_3 = s_4 = 0 (useless)
ii) s_1 = s_2 = s_3 = 1, s_4 = 0 --> [1,1,1,0] solution

However, we can condesne this process by eliminating repetitive equations so that the remaining equations are:
1. s_1 (+) s_2 = 0 
2. s_2 (+) s_3 (+) s_4 = 0 
3. s_1 (+) s_3 = 0

So, we need n-1 equations, and in this case, 4 - 1 = 3 equatiins